# Chapter 11: Simple Sorts

The programs from Chapter 11 of *Data Structures and Algorithms in Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name. Benchmark timings differ on every computer and every run.

Run the cells from top to bottom (**Run > Run All Cells**). The programs import each other's files, so keep this notebook in its chapter folder.

## Three Simple Sorts

`simple_sorts.py`

In [1]:
"""Three simple sorts. Each sorts the list in place, in quadratic time."""


def selection_sort(items):
    """Find the smallest remaining item and swap it into place."""
    n = len(items)
    for i in range(n - 1):
        smallest = i
        for j in range(i + 1, n):
            if items[j] < items[smallest]:
                smallest = j
        items[i], items[smallest] = items[smallest], items[i]


def insertion_sort(items):
    """Grow a sorted part on the left, inserting each new item into it."""
    for i in range(1, len(items)):
        x = items[i]
        j = i - 1
        while j >= 0 and items[j] > x:  # shift bigger items right
            items[j + 1] = items[j]
            j -= 1
        items[j + 1] = x


def bubble_sort(items):
    """Swap neighbors that are out of order; stop when a pass swaps none."""
    n = len(items)
    for end in range(n - 1, 0, -1):
        swapped = False
        for j in range(end):
            if items[j] > items[j + 1]:
                items[j], items[j + 1] = items[j + 1], items[j]
                swapped = True
        if not swapped:
            return

## Counting the Work

`count_work.py`

In [2]:
"""Count the comparisons each sort makes on 1,000 items, arranged
three ways. Insertion sort adapts: nearly sorted input is cheap."""

import random

from simple_sorts import bubble_sort, insertion_sort, selection_sort


class Counted:
    """A number that counts every comparison made with it."""

    comparisons = 0

    def __init__(self, x):
        self.x = x

    def __lt__(self, other):
        Counted.comparisons += 1
        return self.x < other.x

    def __gt__(self, other):
        Counted.comparisons += 1
        return self.x > other.x


rng = random.Random(2026)
n = 1000
nearly = list(range(n))
for _ in range(10):  # swap ten random pairs of neighbors
    i = rng.randrange(n - 1)
    nearly[i], nearly[i + 1] = nearly[i + 1], nearly[i]
inputs = {
    "random": rng.sample(range(n), n),
    "nearly sorted": nearly,
    "reversed": list(range(n, 0, -1)),
}
print(f"{'':>15}" + "".join(f"{name:>15}" for name in inputs))
for sort in [selection_sort, insertion_sort, bubble_sort]:
    row = f"{sort.__name__:>15}"
    for data in inputs.values():
        items = [Counted(x) for x in data]
        Counted.comparisons = 0
        sort(items)
        row += f"{Counted.comparisons:>15,}"
    print(row)

                        random  nearly sorted       reversed


 selection_sort        499,500        499,500        499,500
 insertion_sort        258,269          1,009        499,500


    bubble_sort        496,799          2,994        499,500


## Stability

`stability.py`

In [3]:
"""A stable sort keeps items with equal keys in their original order."""


def insertion_sort_by(items, key):
    for i in range(1, len(items)):
        x = items[i]
        j = i - 1
        while j >= 0 and key(items[j]) > key(x):  # > not >=: stable
            items[j + 1] = items[j]
            j -= 1
        items[j + 1] = x


def selection_sort_by(items, key):
    for i in range(len(items) - 1):
        m = min(range(i, len(items)), key=lambda j: key(items[j]))
        items[i], items[m] = items[m], items[i]  # this swap can jump


people = [("Ada", 30), ("Bo", 25), ("Cy", 30), ("Di", 25)]
for sort in (insertion_sort_by, selection_sort_by):
    items = list(people)
    sort(items, key=lambda p: p[1])  # by age
    print(f"{sort.__name__:>18}: {items}")

 insertion_sort_by: [('Bo', 25), ('Di', 25), ('Ada', 30), ('Cy', 30)]
 selection_sort_by: [('Bo', 25), ('Di', 25), ('Cy', 30), ('Ada', 30)]


## Timing

`bench_simple.py`

In [4]:
"""The three simple sorts on random lists, and insertion sort on a
nearly sorted list."""

import random

import bench
from simple_sorts import bubble_sort, insertion_sort, selection_sort

rng = random.Random(2026)


def shuffled(n):
    return (rng.sample(range(n), n),)


def nearly_sorted(n):
    items = list(range(n))
    for _ in range(n // 100):  # one neighbor swap per 100 items
        i = rng.randrange(n - 1)
        items[i], items[i + 1] = items[i + 1], items[i]
    return (items,)


sizes = [250, 500, 1_000, 2_000]
results = {
    "bubble": bench.doubling(bubble_sort, shuffled, sizes, fresh=True),
    "selection": bench.doubling(
        selection_sort, shuffled, sizes, fresh=True
    ),
    "insertion": bench.doubling(
        insertion_sort, shuffled, sizes, fresh=True
    ),
    "nearly sorted": bench.doubling(  # insertion sort
        insertion_sort, nearly_sorted, sizes, fresh=True
    ),
}
bench.table(results)
bench.chart(results, "ch11_simple.png", "Simple sorts")

         n          bubble       selection       insertion   nearly sorted
       250         1.25 ms          566 us          598 us         12.4 us
       500    5.08 ms x4.1    2.45 ms x4.3    2.38 ms x4.0    23.5 us x1.9
     1,000    19.3 ms x3.8    9.34 ms x3.8    8.93 ms x3.8    40.6 us x1.7
     2,000    91.2 ms x4.7      51 ms x5.5    40.8 ms x4.6      95 us x2.3
